# Clase 9: Exploración inicial y visualización de datos

**Objetivos de hoy:**
1. **Persistencia de Datos:** Concluir la fase de limpieza eliminando registros nulos en ingresos (`Revenue`) y exportar a un archivo `.csv`.
2. **Entender las Herramientas:** Comprender la diferencia arquitectónica entre **Matplotlib** (el lienzo base) y **Seaborn** (la capa estadística de alto nivel).
3. **Exploración Visual:** Aprender a elegir el gráfico correcto según el tipo de datos (categóricos vs. numéricos).


## Cargamos los dataframes de la clase anterior

In [ ]:
import pandas as pd

# Cargamos el dataset estructurado que acabamos de limpiar
df_movies = pd.read_csv('movies_limpio.csv')
print("Dataset listo para la exploración visual. Primeras 3 filas:")
df_movies.head(3)


## Arquitectura de visualización en Python

Para dominar la visualización de datos, debemos entender que no compiten entre sí, sino que se complementan:

* **Matplotlib (`matplotlib.pyplot`):** Es la biblioteca de bajo nivel. Funciona como un **lienzo en blanco**. Nosotros tenemos que definir manualmente el título, los ejes, los colores y las etiquetas. Es sumamente personalizable, pero requiere más líneas de código.
* **Seaborn (`seaborn`):** Es una biblioteca de alto nivel construida **encima** de Matplotlib. Está diseñada específicamente para gráficos estadísticos. Entiende directamente los DataFrames de Pandas, aplica paletas de colores profesionales de forma automática y genera gráficos complejos (como mapas de calor) en una sola línea de código.

In [ ]:
%pip install seaborn

In [1]:
# Importación de librerías
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración global de Seaborn para que todos los gráficos tengan un diseño limpio y legible
sns.set_theme(style="whitegrid")


In [ ]:
df_movies.describe()

In [ ]:
df_movies.tail()

In [ ]:
# Asegurar tipos numéricos
df_movies['Runtime'] = pd.to_numeric(df_movies['Runtime'], errors='coerce')
df_movies['Year'] = pd.to_numeric(df_movies['Year'], errors='coerce')
df_movies['Score'] = pd.to_numeric(df_movies['Score'], errors='coerce')
df_movies['Revenue'] = pd.to_numeric(df_movies['Revenue'], errors='coerce')

In [ ]:
# ¿Cuántas filas y columnas tiene el dataset?
df_movies.shape

# ¿Qué columnas hay y qué tipo de datos tienen?
df_movies.info()

### 1.1 Visualización muy básica. Sin configurar con los parámetros de matplotlib.

Pandas cuenta con la función plot() que nos muestra un gráfico sencillo. 
Basta con colocar plot() al final de una serie de datos para crear de manera rápida una gráfica. Podemos colocar entre paréntesis con 
el parámetro kind el tipo de gráfico que deseamos:

|kind	|Tipo de gráfico|
|-------|--------------------------------|
|'line'	|Gráfico de líneas (por defecto)|
|'bar'	|Gráfico de barras verticales|
|'barh'	|Gráfico de barras horizontales|
|'hist'	|Histograma|
|'box'	|Diagrama de caja (boxplot)|
|'kde'	|Estimación de densidad Kernel|
|'density'	|Alias de 'kde'|
|'area'	|Gráfico de área|
|'pie'	|Gráfico de pastel (pie chart)|
|'scatter'	|Diagrama de dispersión (requiere x e y)|
|'hexbin'	|Diagrama hexagonal de densidad (requiere x e y)|


In [ ]:
# Visualización más básica y sin configurar del número de películas por año.
df_movies['Year'].value_counts().sort_index().plot(kind='bar')

In [ ]:
# ¿Cuántas películas hay por año?
df_movies['Year'].value_counts().sort_index().plot(kind='line')
# Esta gráfica sencilla puede configurarse usando matplotlib con las siguientes instrucciones (Recuerda que plt es matplotlib)
plt.title("Películas por año")
plt.xlabel("Año")
plt.ylabel("Cantidad")
plt.show()

In [ ]:
df_movies['Director'].value_counts().head(10).plot()

In [ ]:
df_movies.groupby("Director")["Revenue"].mean().sort_values(ascending=False).plot()
plt.title("Revenue promedio por director")
plt.xlabel("Director")
plt.ylabel("Revenue")
plt.show()

## Visualización básica y categórica con Matplotlib


### Análisis de una sola variable (univariable)

* **Histogramas:** Se usan para **variables numéricas continuas** (ej. la duración en minutos). Nos permite ver la distribución: dónde se concentran la mayoría de las películas y si hay sesgos.
* **Gráficos de Barras:** Se usan para **variables categóricas** (ej. Directores o Géneros). Cuenta la frecuencia o compara magnitudes entre categorías discretas.

In [ ]:
# Crear el lienzo con Matplotlib
plt.figure(figsize=(10, 5))

# Graficar un histograma de la columna 'Runtime' (duración en minutos)
plt.hist(df_movies['Runtime'], bins=20, color='skyblue', edgecolor='black')

# Personalización de etiquetas y título
plt.title('Distribución de la duración de las películas', fontsize=14, fontweight='bold')
plt.xlabel('Duración en minutos', fontsize=12)
plt.ylabel('Frecuencia (Cantidad de películas)', fontsize=12)

# Mostrar el gráfico
plt.show()

In [ ]:
# Crear el lienzo
plt.figure(figsize=(10, 6))

# 1. Obtenemos los 10 directores más frecuentes
top_directores = df_movies['Director'].value_counts().head(10)

# 2. Creamos el gráfico de barras horizontales aplicando la corrección de Seaborn
sns.barplot(
    x=top_directores.values, 
    y=top_directores.index, 
    hue=top_directores.index,   # Asigna el color a cada director en el eje Y
    palette='viridis', 
    legend=False                # Desactiva la leyenda innecesaria
)

# Personalización del gráfico
plt.title('Top 10 de directores con mayor cantidad de películas (Seaborn)', fontsize=14, fontweight='bold')
plt.xlabel('Cantidad de películas', fontsize=12)
plt.ylabel('Director', fontsize=12)

# Mostrar gráfico
plt.show()

### Análisis bivariable y multivariable (Cruzar datos)

* **Gráfico de dispersión (Scatter Plot):** Ideal para ver la relación entre dos **variables numéricas**. Permite identificar visualmente si una variable afecta a la otra (correlación) y detectar valores atípicos (*outliers*).
* **Diagrama de caja (Boxplot):** Muestra la distribución de los datos a través de sus cuartiles. Es la mejor herramienta para visualizar la simetría, la dispersión y los valores extremos de forma geométrica.

In [ ]:
# Creamos un lienzo dividido en 1 fila y 2 columnas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Gráfico de Dispersión (Usa tu sintaxis de Pandas/Matplotlib)
df_movies.plot(kind='scatter', x='Runtime', y='Revenue', ax=axes[0], color='purple', alpha=0.6)
axes[0].set_title('Relación: Duración vs Ingresos', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Duración (Minutos)')
axes[0].set_ylabel('Ingresos (Millones)')

# 2. Diagrama de Caja (Usa tu sintaxis con Seaborn en el segundo eje)
sns.boxplot(data=df_movies, x='Revenue', ax=axes[1], color='lightgreen')
axes[1].set_title('Distribución Geométrica de Ingresos (Outliers)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Ingresos en Millones')

plt.tight_layout()
plt.show()

### Análisis de evolución temporal (Líneas de tendencia)

Cuando una de nuestras variables numéricas es el **Año** o el **Tiempo**, el gráfico por excelencia es el **Gráfico de Líneas**. 
No nos interesa ver puntos dispersos, sino la *dirección* que toman los datos: ¿las películas han mejorado con los años o su calidad ha decaído? ¿La taquilla va en aumento?

Para esto, agrupamos primero los datos por año (`groupby`) y calculamos el promedio de la variable cuantitativa.

In [ ]:

# 1. Configurar el lienzo
plt.figure(figsize=(12, 5))

# 2. Calcular el promedio de Score por año de forma limpia
df_tendencia = df_movies.groupby("Year")["Score"].mean().reset_index()

# 3. Graficar la línea usando Seaborn para mejorar el diseño visual
sns.lineplot(
    data=df_tendencia, 
    x="Year", 
    y="Score", 
    marker="o",          # Añade círculos en cada año para marcar el dato exacto
    linewidth=2.5,       # Hace la línea un poco más gruesa y visible
    color="#e74c3c"      # Un color rojo elegante para resaltar la tendencia
)

# 4. Personalización del lienzo (Matplotlib)
plt.title("Evolución del Puntaje Promedio de las Películas por Año", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Año de Estreno", fontsize=12)
plt.ylabel("Puntaje Promedio (Score)", fontsize=12)

# Forzar a que los años en el eje X se muestren como enteros y no decimales
plt.locator_params(axis='x', integer=True)

# Mostrar la cuadrícula de fondo de manera sutil para facilitar la lectura del puntaje
plt.grid(True, linestyle="--", alpha=0.6)

plt.show()

In [ ]:
# Distribución del tiempo de duración de películas
df_movies['Runtime'].hist(bins=20)
plt.title("Distribución de duración de películas")
plt.xlabel("Minutos")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
# Dispersión entre Score y Revenue
sns.scatterplot(data=df_movies, x="Score", y="Revenue")
plt.title("Score vs Revenue")
plt.show()

In [ ]:
# Boxplot de Score por género (usamos solo el primer género si hay múltiples)
sns.boxplot(data=df_movies, x="GeneroPrincipal", y="Score")
plt.title("Distribución del Score por género")
plt.xticks(rotation=45)
plt.show()

### El mapa de calor (Heatmap)
El mapa de calor es una representación gráfica de una **matriz de correlación**. 
La correlación es un valor numérico entre `-1` y `1` que mide la fuerza de la relación lineal entre variables:
* Cercano a `1`: Relación positiva fuerte (si una sube, la otra también).
* Cercano a `0`: No hay relación aparente.
* Cercano a `-1`: Relación negativa (si una sube, la otra baja).

In [ ]:
# Correlación entre variables numéricas
numericas = df_movies[["Score", "Metascore", "Vote", "Revenue", "Runtime"]]
correlaciones = numericas.corr()

sns.heatmap(correlaciones, annot=True, cmap="coolwarm")
plt.title("Mapa de calor de correlación")
plt.show()

# Parte 2: Limpieza y cálculo de Decada


### --- LIMPIEZA Y PREPARACIÓN ---
Limpiemos y creemos la variable Decada para pasar a la parte gráfica.


In [ ]:
df_movies.columns

In [ ]:

# Crear columna de década
df_movies = df_movies[df_movies['Year'].notna()]
df_movies['Year'] = df_movies['Year'].astype(int)
df_movies['Decada'] = (df_movies['Year'] // 10) * 10


In [ ]:
df_movies.head(2)

In [ ]:
# Esta instrucción nos ayuda a contar el número de datos faltantes por columna
df_movies.isna().sum()

In [ ]:
# Aquí dejo algunas instrucciones en caso de querer borrar las filas que tienen un faltante en la columna especificada
# Eliminar valores faltantes en duración para algunos análisis
#df_movies = df_movies[df_movies['Runtime'].notna()]

# Eliminar valores faltantes en Revenue para algunos análisis
#df_movies = df_movies[df_movies['Revenue'].notna()]

# Eliminar valores faltantes en Revenue para algunos análisis
#df_movies = df_movies[df_movies['Director'].notna()]


In [ ]:

# --- ANÁLISIS 1: Cantidad de películas por década ---
pelis_por_decada = df_movies['Decada'].value_counts().sort_index()

plt.figure(figsize=(10,5))   # si no pusimos al principio de manera global el tamaño de la imagen podemos hacerlo para cada imagen con esta instrucción
pelis_por_decada.plot(kind='bar')
plt.title('Número de películas por década')
plt.xlabel('Década')
plt.ylabel('Cantidad de películas')
plt.grid(axis='y')
plt.show()


In [ ]:

# --- ANÁLISIS 2: Duración promedio por década ---
plt.figure(figsize=(10,5))
df_movies.groupby('Decada')['Runtime'].mean().plot(kind='line', marker='o')
plt.title('Duración promedio por década')
plt.xlabel('Década')
plt.ylabel('Duración promedio (minutos)')
plt.grid()
plt.show()


In [ ]:

# --- ANÁLISIS 3: Boxplot por década ---
plt.figure(figsize=(12,6))
sns.boxplot(data=df_movies, x='Decada', y='Runtime')
plt.xticks(rotation=45)
plt.title('Distribución de duración de películas por década')
plt.show()


In [ ]:

# --- ANÁLISIS 4: Regresión lineal ---
# Seaborn cuenta con la gráfica regplot, que una vez listos nuestros datos puede recibir 
# aquellas dos columnas de las cuales necesite hacer una regresión
sns.regplot(data=df_movies, x='Year', y='Runtime', scatter_kws={'alpha':0.3})
plt.title('Relación entre año y duración de películas')
plt.xlabel('Año')
plt.ylabel('Duración (minutos)')
plt.grid()
plt.show()

In [ ]:

# --- ANÁLISIS 5: Mapa de calor de correlación ---

# Correlación entre variables numéricas
numericas = df_movies[["Decada","Score", "Metascore", "Vote", "Revenue", "Runtime"]]
correlaciones = numericas.corr()
plt.figure(figsize=(6,5))
sns.heatmap(correlaciones, annot=True, cmap="cool") # El parámetro cmap cotiene lso colores, se buscan como colormap en la página de matplotlib
plt.title("Correlación entre variables numéricas")
#si queremos guardar esta imagen lo hacemos con 
plt.savefig("correlacion.png")
plt.show()

**Figuras con más de una gráfica por imagen**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
df_movies['Runtime'].plot(kind='hist', ax=axes[0], bins=20, color='lightblue')
axes[0].set_title('Histograma de duración (Runtime)')

# KDE
df_movies['Runtime'].plot(kind='kde', ax=axes[1], color='darkblue')
axes[1].set_title('Curva de densidad (KDE) de duración')

plt.tight_layout()
plt.show()


In [ ]:

# Calculamos el conteo de películas por año
conteo_por_año = df_movies['Year'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Línea
conteo_por_año.plot(kind='line', ax=axes[0], marker='o')
axes[0].set_title('Películas por año (línea)')

# Barras
conteo_por_año.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Películas por año (barras)')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
df_movies.plot(kind='scatter', x='Runtime', y='Score', ax=axes[0])
axes[0].set_title('Duración vs Score')

# Boxplot por Score
sns.boxplot(data=df_movies, y='Score', ax=axes[1])
axes[1].set_title('Distribución del Score')

plt.tight_layout()
plt.show()



## Ejercicios:

**Instrucciones:** Resuelvan los siguientes tres bloques de código y análisis utilizando el DataFrame `df_movies`. Recuerden aplicar las buenas prácticas de diseño vistas en clase (títulos legibles, etiquetas en los ejes y colores adecuados). Al finalizar, redacten sus conclusiones en las celdas de Markdown correspondientes.

---

### Ejercicio 1: Modificación de distribución (Variable Cuantitativa)
Tomen como base el histograma de la duración de las películas (`Runtime`) visto en la sección 1 y realicen las siguientes modificaciones:
1. Incrementen el número de contenedores a `bins=35`.
2. Cambien el color de las barras a un tono de su elección (pueden usar nombres de colores en inglés o códigos hexadecimales) y agreguen un color de borde (`edgecolor`) diferente.
3. **Pregunta de Reflexión (Markdown):** Al aumentar el número de `bins`, ¿la distribución de los datos se vuelve más detallada o más ruidosa? ¿Qué rango de duración es ahora el más evidente como el "estándar" en la industria del cine?

---

### Ejercicio 2: Análisis de relación (Cruzar variables cuantitativas)
Un productor de cine asegura que *"las películas con mejores calificaciones del público (`Rating`) siempre son las que más dinero recaudan en taquilla (`Revenue`)"*. Como científicos de datos, verifiquen esta hipótesis:
1. Utilicen Seaborn (`sns.scatterplot`) para generar un gráfico de dispersión que cruce la variable `Rating` en el eje X y la variable `Revenue` en el eje Y.
2. Personalicen el lienzo con Matplotlib agregando un título en negritas y nombres claros en los ejes.
3. **Pregunta de Reflexión (Markdown):** Observando la nube de puntos, ¿existe una relación lineal clara que apoye la teoría del productor? ¿Qué pasa con las películas que tienen calificaciones excelentes (mayores a 8.0) pero ingresos bajos?

---

### Ejercicio 3: Interpretación de la matriz de correlación (Análisis multivariable)
Observen detenidamente el **Mapa de Calor (Heatmap)** que se generó en la sección de análisis multivariable y respondan a las siguientes preguntas en una celda de texto:
1. ¿Cuál es el par de variables numéricas que presenta la correlación positiva más fuerte en todo el dataset? ¿Qué significa esto en el mundo real?
2. Si quisiéramos predecir el éxito financiero de una película (`Revenue`), ¿deberíamos fijarnos más en la **duración de la película (`Runtime`)** o en la **cantidad de votos recibidos (`Votes`)**? Justifiquen su respuesta utilizando los valores numéricos exactos de la matriz de correlación.